# DesertMap Data Cleaning and Feature Engineering

This notebook prepares the USDA Food Access Research Atlas data for downstream modeling. The target is `LILATracts_1And10`, and the exported dataset becomes the single source of truth for later modeling notebooks.

## Section 0: Setup

This section imports the analysis libraries, applies the project plotting defaults, and loads the raw USDA dataset. It also defines the modeling features, target column, and tract identifier used throughout the cleaning workflow.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook')
np.random.seed(42)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = PROJECT_ROOT / 'data/raw/FoodAccessResearchAtlasData2019.csv'
PROCESSED_DIR = PROJECT_ROOT / 'data/processed'
FIGURE_DIR = PROJECT_ROOT / 'outputs/figures'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_FEATURES = [
    'MedianFamilyIncome',
    'PovertyRate',
    'pct_nhblack10',
    'pct_hisp10',
    'pct_nhwhite10',
    'pct_hunv10',
    'pct_snap16',
    'Pop2010',
    'Urban',
]
TARGET_COL = 'LILATracts_1And10'
CONTINUOUS_FEATURES = [feature for feature in MODEL_FEATURES if feature != 'Urban']
FIPS_CANDIDATES = ['CensusTract', 'TractFIPS', 'FIPS']

if not DATA_PATH.exists():
    raise FileNotFoundError(f'Missing required data file: {DATA_PATH}')

food_access = pd.read_csv(DATA_PATH, low_memory=False)

missing_required_columns = [col for col in MODEL_FEATURES + [TARGET_COL] if col not in food_access.columns]
if missing_required_columns:
    raise KeyError(f'Missing required columns: {missing_required_columns}')

fips_col = next((col for col in FIPS_CANDIDATES if col in food_access.columns), None)
if fips_col is None:
    raise KeyError(f'No FIPS identifier column found. Tried: {FIPS_CANDIDATES}')

print(f'Dataset shape: {food_access.shape[0]:,} rows x {food_access.shape[1]:,} columns')
print(f'FIPS identifier column: {fips_col}')
print(f'Target column: {TARGET_COL}')
print('Model features:')
for feature in MODEL_FEATURES:
    print(f'  - {feature}')

### Feature Derivation

The USDA Food Access Research Atlas stores racial and demographic data as tract-level counts 
(`TractBlack`, `TractHispanic`, etc.) rather than percentages. 
This cell derives the percentage equivalents used throughout the project. 
Tracts where `OHU2010` is zero receive NaN for household-rate features; 
the cleaning notebook handles these with median imputation.

In [ ]:
# Derive percentage features from tract count columns (same logic as 01_eda.ipynb)
ohu = food_access['OHU2010'].replace(0, float('nan'))
pop = food_access['Pop2010']

food_access['pct_nhblack10'] = (food_access['TractBlack'] / pop * 100).clip(upper=100)
food_access['pct_hisp10']    = (food_access['TractHispanic'] / pop * 100).clip(upper=100)
food_access['pct_nhwhite10'] = (food_access['TractWhite'] / pop * 100).clip(upper=100)
food_access['pct_hunv10']    = (food_access['TractHUNV'] / ohu * 100).clip(upper=100)
food_access['pct_snap16']    = (food_access['TractSNAP'] / ohu * 100).clip(upper=100)


## Section 1: Target Variable Integrity

This section removes rows without a known food desert label and validates that the target is binary. The target is never imputed because missing labels cannot support supervised training.

In [ ]:
initial_rows = len(food_access)
clean_data = food_access.dropna(subset=[TARGET_COL]).copy()
dropped_target_rows = initial_rows - len(clean_data)

print(f'Rows dropped for missing {TARGET_COL}: {dropped_target_rows:,}')

target_values = set(clean_data[TARGET_COL].dropna().unique())
if not target_values.issubset({0, 1, 0.0, 1.0}):
    raise ValueError(f'{TARGET_COL} must contain only 0 and 1. Found: {sorted(target_values)}')

class_balance = clean_data[TARGET_COL].value_counts(normalize=True).sort_index().mul(100).round(2)
class_counts = clean_data[TARGET_COL].value_counts().sort_index()
class_summary = pd.DataFrame({
    'count': class_counts,
    'percentage': class_balance,
})

print('Class balance after dropping missing targets:')
print(class_summary.to_string())

## Section 2: Missing Value Analysis and Imputation

This section profiles feature-level missingness and applies deterministic imputation rules. Continuous numeric features use medians because income, rates, and population variables can be skewed; the binary `Urban` flag uses the mode.

In [ ]:
before_missing = clean_data[MODEL_FEATURES].isna().sum()
imputation_records = []

for feature in MODEL_FEATURES:
    missing_count = int(clean_data[feature].isna().sum())
    missing_pct = missing_count / len(clean_data) * 100
    dtype = clean_data[feature].dtype

    if feature == 'Urban':
        mode_values = clean_data[feature].mode(dropna=True)
        if mode_values.empty:
            raise ValueError('Urban has no non-missing values available for mode imputation.')
        fill_value = mode_values.iloc[0]
        strategy = f'mode ({fill_value})'
    else:
        fill_value = clean_data[feature].median()
        strategy = f'median ({fill_value:.3f})'

    print(f'{feature} | dtype: {dtype} | missing: {missing_count:,} ({missing_pct:.2f}%) | strategy: {strategy}')
    clean_data[feature] = clean_data[feature].fillna(fill_value)
    imputation_records.append({
        'feature': feature,
        'dtype': str(dtype),
        'missing_before': missing_count,
        'missing_pct_before': round(missing_pct, 2),
        'strategy': strategy,
    })

after_missing = clean_data[MODEL_FEATURES].isna().sum()
remaining_missing = after_missing[after_missing > 0]
if not remaining_missing.empty:
    raise AssertionError(f'Missing values remain after imputation: {remaining_missing.to_dict()}')

missing_summary = pd.DataFrame(imputation_records).set_index('feature')
missing_summary['missing_after'] = after_missing

print('\nBefore/after missingness summary:')
print(missing_summary.to_string())

## Section 3: Outlier Detection and Capping

This section applies IQR-based capping to continuous modeling features only. The binary `Urban` feature and the target label are excluded from outlier handling.

In [ ]:
pre_cap_data = clean_data[CONTINUOUS_FEATURES].copy()
capping_records = []

for feature in CONTINUOUS_FEATURES:
    q1 = clean_data[feature].quantile(0.25)
    q3 = clean_data[feature].quantile(0.75)
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr
    lower_count = int((clean_data[feature] < lower_fence).sum())
    upper_count = int((clean_data[feature] > upper_fence).sum())

    clean_data[feature] = clean_data[feature].clip(lower=lower_fence, upper=upper_fence)
    capping_records.append({
        'feature': feature,
        'lower_fence': lower_fence,
        'upper_fence': upper_fence,
        'capped_lower': lower_count,
        'capped_upper': upper_count,
    })

capping_summary = pd.DataFrame(capping_records)
print('IQR capping summary:')
print(capping_summary.to_string(index=False, float_format=lambda value: f'{value:.3f}'))

## Section 4: Feature Distribution Check (Post-Cleaning)

This section compares skewness before and after capping and visualizes post-cleaning feature distributions by food desert label. These checks help verify that preprocessing reduced extreme values without changing the target.

In [ ]:
skewness_records = []

for feature in CONTINUOUS_FEATURES:
    skew_before = stats.skew(pre_cap_data[feature], nan_policy='omit')
    skew_after = stats.skew(clean_data[feature], nan_policy='omit')
    skewness_records.append({
        'feature': feature,
        'skew_before_capping': skew_before,
        'skew_after_capping': skew_after,
    })

skewness_summary = pd.DataFrame(skewness_records)
print('Skewness before and after capping:')
print(skewness_summary.to_string(index=False, float_format=lambda value: f'{value:.3f}'))

plot_data = clean_data[MODEL_FEATURES + [TARGET_COL]].copy()
plot_data['Food desert label'] = plot_data[TARGET_COL].map({0: 'Not food desert', 1: 'Food desert'})
palette = {'Not food desert': '#4C78A8', 'Food desert': '#F58518'}

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for ax, feature in zip(axes, MODEL_FEATURES):
    sns.boxplot(
        data=plot_data,
        x='Food desert label',
        y=feature,
        hue='Food desert label',
        palette=palette,
        dodge=False,
        ax=ax,
    )
    title_suffix = ' (binary)' if feature == 'Urban' else ''
    ax.set_title(f'{feature}{title_suffix} by food desert label')
    ax.set_xlabel('Food desert label')
    ax.set_ylabel(f'{feature}{title_suffix}')
    ax.tick_params(axis='x', rotation=20)
    if ax.get_legend() is not None:
        ax.get_legend().remove()

handles, labels = axes[0].get_legend_handles_labels()
if not handles:
    handles = [plt.Line2D([0], [0], color=color, lw=4) for color in palette.values()]
    labels = list(palette.keys())
fig.legend(handles, labels, title='Food desert label', loc='upper center', ncol=2, frameon=True)
fig.suptitle('Post-cleaning feature distributions by food desert label', y=1.03)
fig.tight_layout(rect=[0, 0, 1, 0.98])

boxplot_path = FIGURE_DIR / '02_post_cleaning_boxplots.png'
fig.savefig(boxplot_path, dpi=150, bbox_inches='tight')
plt.show()

print(f'Saved figure to: {boxplot_path.resolve()}')

## Section 5: Correlation with Target (Post-Cleaning)

This section calculates Pearson correlations between each modeling feature and the food desert target. Very weak correlations are flagged as candidates for later model-selection review, not removed here.

In [ ]:
correlation_records = []

for feature in MODEL_FEATURES:
    feature_values = clean_data[feature]
    if feature_values.nunique(dropna=False) <= 1:
        correlation = np.nan
        p_value = np.nan
    else:
        correlation, p_value = stats.pearsonr(feature_values, clean_data[TARGET_COL])

    correlation_records.append({
        'feature': feature,
        'pearson_correlation': correlation,
        'p_value': p_value,
    })

correlation_summary = pd.DataFrame(correlation_records).sort_values('pearson_correlation', ascending=False)
print('Pearson correlation with target:')
print(correlation_summary.to_string(index=False, float_format=lambda value: f'{value:.4f}'))

weak_features = correlation_summary.loc[
    correlation_summary['pearson_correlation'].abs() < 0.05,
    'feature',
].tolist()

if weak_features:
    print('\nFeatures with |correlation| < 0.05; candidates for removal in later notebooks:')
    for feature in weak_features:
        print(f'  - {feature}')
else:
    print('\nNo features have |correlation| < 0.05.')

## Section 6: Export Clean Dataset

This section writes the final modeling dataset with the tract identifier, model features, and target label. The exported CSV is the single source of truth for downstream modeling notebooks.

In [ ]:
fips_col = next((col for col in FIPS_CANDIDATES if col in clean_data.columns), None)
if fips_col is None:
    raise KeyError(f'No FIPS identifier column found. Tried: {FIPS_CANDIDATES}')

final_columns = [fips_col] + MODEL_FEATURES + [TARGET_COL]
modeling_data = clean_data[final_columns].copy()

output_path = PROCESSED_DIR / 'modeling_data_clean.csv'
modeling_data.to_csv(output_path, index=False)

final_class_balance = modeling_data[TARGET_COL].value_counts(normalize=True).sort_index().mul(100).round(2)
food_desert_rate = modeling_data[TARGET_COL].mean() * 100

print(f'Final shape: {modeling_data.shape[0]:,} rows x {modeling_data.shape[1]:,} columns')
print('Final class balance:')
print(final_class_balance.to_string())
print(f'Saved file: {output_path.resolve()}')

print('\nClean dataset saved.')
print(f'Rows: {len(modeling_data):,} | Features: {len(MODEL_FEATURES)} | Target: {TARGET_COL}')
print(f'Food desert rate: {food_desert_rate:.1f}%')
print('Next: notebooks/03_baseline_model.ipynb')